# Multi-Model Synthetic Data Generation Platform

**Hugging Face Transformers + Google Colab + Gradio**

This notebook builds an industry-independent synthetic-data generator. The included demonstration prompt targets a Nigerian solar installation business, but users can replace it with aviation, retail, fintech, healthcare administration, or another business scenario.

### What it does
- Loads open-weight instruction models from Hugging Face.
- Generates fictional, structured records using multiple prompt strategies.
- Runs models **sequentially** to reduce GPU-memory pressure.
- Validates the requested JSON schema and removes exact duplicates.
- Compares models and exports CSV/JSON.
- Provides a Gradio web interface.

### Important limitations
- Synthetic records are fictional and must not be presented as real customer or operational data.
- This notebook does not establish that synthetic data represents a real population or is safe for training a safety-critical system.
- Colab GPU availability and model download speed depend on your account/session.
- The notebook has been authored for Colab, but you must run it in your own runtime to verify model downloads and GPU compatibility.


## 1. Select a Colab runtime

In Colab, choose **Runtime → Change runtime type → T4 GPU** (or another available GPU). A GPU is strongly recommended, but availability is not guaranteed.

This notebook uses small models and loads only one model at a time. If you receive CUDA out-of-memory errors, reduce the batch size in the generation settings or switch to the smallest model.


## Free Colab T4 settings (recommended)

This notebook is tuned for a **free Colab T4 session (typically 16 GB VRAM when a T4 is allocated)**:

- The selected models are small enough to test individually on a T4 in FP16.
- Only one model is loaded at a time. Comparison unloads each model before loading the next.
- Start with **one model, 6 records, batch size 2, and temperature 0.8**.
- For the first model comparison, use 3–5 records per model.
- Colab may disconnect or reclaim its GPU. Download CSV/JSON outputs before resetting or closing the session.
- A runtime reset clears RAM/VRAM and temporary `/content` files. Rerun cells from the top.
- Model weights may need to download again after a factory reset or a new runtime.


In [ ]:
# Install dependencies
%pip -q install -U transformers accelerate gradio pandas huggingface_hub sentencepiece


In [ ]:
import os
import re
import gc
import json
import time
import tempfile
from pathlib import Path
from typing import Any

import pandas as pd
import torch
import gradio as gr
from transformers import AutoTokenizer, AutoModelForCausalLM

print("Python environment ready")
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. You can still test with a small model, but generation may be slow.")


In [ ]:
def show_gpu_memory():
    if not torch.cuda.is_available():
        print("CUDA is not available in this runtime.")
        return
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name}")
    print(f"Total VRAM: {props.total_memory / (1024 ** 3):.2f} GB")
    print(f"Allocated: {torch.cuda.memory_allocated(0) / (1024 ** 3):.2f} GB")
    print(f"Reserved: {torch.cuda.memory_reserved(0) / (1024 ** 3):.2f} GB")

show_gpu_memory()


## 2. Configure the models

These are small instruction/chat models suitable for experimentation. They differ in size and capability; do not assume the largest or newest model will always produce the best structured output.

If a repository changes, is unavailable, or requires additional access, replace its model ID with another openly accessible instruction model. Check each model's licence and usage terms before commercial use.


In [ ]:
MODEL_CATALOG = {
    "Qwen2.5 1.5B Instruct": "Qwen/Qwen2.5-1.5B-Instruct",
    "SmolLM2 1.7B Instruct": "HuggingFaceTB/SmolLM2-1.7B-Instruct",
    "TinyLlama 1.1B Chat": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
}

# The notebook keeps at most one model loaded at a time.
_LOADED_MODEL_ID = None
_TOKENIZER = None
_MODEL = None

def release_model():
    """Free the currently loaded model and cached GPU memory."""
    global _LOADED_MODEL_ID, _TOKENIZER, _MODEL

    if _MODEL is not None:
        del _MODEL
    if _TOKENIZER is not None:
        del _TOKENIZER

    _MODEL = None
    _TOKENIZER = None
    _LOADED_MODEL_ID = None

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def load_model(model_id: str):
    """Load one Hugging Face causal language model."""
    global _LOADED_MODEL_ID, _TOKENIZER, _MODEL

    if _MODEL is not None and _LOADED_MODEL_ID == model_id:
        return _TOKENIZER, _MODEL

    release_model()
    print(f"Loading {model_id} ...")

    tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)

    # Let Transformers choose an appropriate dtype for the available hardware.
    # device_map='auto' places the model on the available GPU/CPU resources.
    model_kwargs = {"device_map": "auto", "low_cpu_mem_usage": True}
    if torch.cuda.is_available():
        # FP16 is supported by NVIDIA T4 and reduces VRAM use.
        model_kwargs["torch_dtype"] = torch.float16
    else:
        model_kwargs["torch_dtype"] = torch.float32

    model = AutoModelForCausalLM.from_pretrained(model_id, **model_kwargs)
    model.eval()

    if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
        tokenizer.pad_token = tokenizer.eos_token

    _LOADED_MODEL_ID = model_id
    _TOKENIZER = tokenizer
    _MODEL = model

    print("Model loaded.")
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / (1024 ** 3)
        reserved = torch.cuda.memory_reserved() / (1024 ** 3)
        print(f"GPU memory allocated: {allocated:.2f} GB | reserved: {reserved:.2f} GB")
    return _TOKENIZER, _MODEL


## 3. Prompt strategies and the output schema

Each record uses the same schema so that results from different models can be compared and exported consistently:

- `record_id`: generated identifier
- `scenario`: short description of the situation
- `text`: the fictional customer message, report, or event
- `attributes`: object containing relevant structured details
- `label`: category or target label requested by the user

The model is asked to return JSON only. The Python code then parses and validates it; invalid output is retried a limited number of times.


In [ ]:
PROMPT_STRATEGIES = {
    "Balanced profiles": (
        "Create a balanced mixture of ordinary, positive, negative, and uncertain cases. "
        "Vary the situations and avoid making every example sound alike."
    ),
    "Edge cases and objections": (
        "Focus on less common, difficult, ambiguous, price-sensitive, dissatisfied, or unusual cases. "
        "Keep examples plausible, but do not invent real people or claim the events actually happened."
    ),
    "Broad scenario diversity": (
        "Cover a wide range of locations, needs, budgets, urgency levels, customer types, and constraints "
        "where relevant to the requested business. Avoid repetitive wording and repeated attribute combinations."
    ),
}

SOLAR_DEMO_PROMPT = """Generate fictional customer enquiries for a Nigerian solar installation company.
Include residential and commercial customers, different Nigerian locations, property types,
energy needs, budget ranges in naira, purchase urgency, and objections such as installation
cost, battery capacity, warranty, maintenance, and reliability.

The records will be used to prototype and test a lead-qualification classifier.
Use varied but plausible examples. Do not use real personal names, phone numbers, addresses,
or other identifying personal information."""

SYSTEM_PROMPT = """You generate fictional synthetic data for software testing and machine-learning prototyping.
Follow the user's requested business context. Never claim the generated records are real.
Return only a valid JSON array. Do not wrap it in Markdown fences or add commentary.

Every array item MUST be a JSON object with exactly these keys:
{
  "record_id": "string",
  "scenario": "string",
  "text": "string",
  "attributes": {"key": "value"},
  "label": "string"
}

Use string values inside attributes for consistent CSV export. Do not include actual personal data.
The label should be a useful category requested or implied by the task.
"""

def build_prompt(description: str, strategy: str, count: int, retry_note: str = "") -> str:
    strategy_text = PROMPT_STRATEGIES[strategy]
    return f"""Business / dataset description:
{description}

Generation strategy:
{strategy_text}

Generate exactly {count} distinct synthetic records.

Required output:
Return a JSON array only. Every record must contain exactly:
record_id, scenario, text, attributes, label.

Use unique record_id values within this batch. Make the records meaningfully different,
not just paraphrases. Use realistic internal consistency, but do not present them as real data.
{retry_note}
"""


## 4. Model inference, JSON extraction and validation

This implementation uses the model's chat template when one is available. It falls back to a plain text prompt if the tokenizer has no chat template.

The generation loop requests small batches instead of asking for a very large JSON array in one response. This makes malformed/truncated outputs easier to recover from.


In [ ]:
REQUIRED_KEYS = {"record_id", "scenario", "text", "attributes", "label"}

def make_chat_prompt(tokenizer, user_prompt: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]

    if getattr(tokenizer, "chat_template", None):
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

    return SYSTEM_PROMPT + "\n\n" + user_prompt + "\n\nJSON:"


@torch.inference_mode()
def generate_text(tokenizer, model, prompt: str, temperature: float, max_new_tokens: int = 1200) -> str:
    formatted_prompt = make_chat_prompt(tokenizer, prompt)
    inputs = tokenizer(formatted_prompt, return_tensors="pt")

    # For device_map='auto', model.device is normally the input device.
    input_device = model.get_input_embeddings().weight.device
    inputs = {key: value.to(input_device) for key, value in inputs.items()}

    generation_kwargs = {
        "max_new_tokens": max_new_tokens,
        "do_sample": temperature > 0,
        "pad_token_id": tokenizer.pad_token_id,
        "eos_token_id": tokenizer.eos_token_id,
    }
    if temperature > 0:
        generation_kwargs["temperature"] = float(temperature)
        generation_kwargs["top_p"] = 0.9

    output = model.generate(**inputs, **generation_kwargs)
    new_tokens = output[0, inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def extract_json_array(raw_text: str):
    """Parse a JSON array, tolerating surrounding explanatory text/fences."""
    cleaned = raw_text.strip()
    cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r"\s*```$", "", cleaned)

    try:
        parsed = json.loads(cleaned)
        if isinstance(parsed, list):
            return parsed
    except json.JSONDecodeError:
        pass

    # Recover the first complete JSON array from the response.
    start = cleaned.find("[")
    if start == -1:
        raise ValueError("No JSON array found in model response.")

    decoder = json.JSONDecoder()
    parsed, _ = decoder.raw_decode(cleaned[start:])
    if not isinstance(parsed, list):
        raise ValueError("Model response did not contain a JSON array.")
    return parsed


def validate_records(items):
    valid = []
    errors = []

    if not isinstance(items, list):
        return [], ["Output is not a JSON list."]

    for index, item in enumerate(items):
        if not isinstance(item, dict):
            errors.append(f"Record {index}: not a JSON object.")
            continue

        missing = REQUIRED_KEYS - set(item.keys())
        if missing:
            errors.append(f"Record {index}: missing keys {sorted(missing)}.")
            continue

        if not isinstance(item["attributes"], dict):
            errors.append(f"Record {index}: attributes must be a JSON object.")
            continue

        record = {key: item[key] for key in REQUIRED_KEYS}
        record["record_id"] = str(record["record_id"]).strip()
        record["scenario"] = str(record["scenario"]).strip()
        record["text"] = str(record["text"]).strip()
        record["label"] = str(record["label"]).strip()
        record["attributes"] = {
            str(k): str(v) for k, v in record["attributes"].items()
        }

        if not record["record_id"] or not record["text"]:
            errors.append(f"Record {index}: record_id and text cannot be empty.")
            continue

        valid.append(record)

    return valid, errors


def record_fingerprint(record):
    """Exact-content duplicate key; not a semantic similarity detector."""
    return json.dumps(
        {
            "scenario": record["scenario"].lower(),
            "text": re.sub(r"\s+", " ", record["text"].lower()).strip(),
            "attributes": record["attributes"],
            "label": record["label"].lower(),
        },
        sort_keys=True,
        ensure_ascii=False,
    )


## 5. Generate records and compare models

The generator:
1. Loads one model.
2. Generates small batches until it reaches the requested record count or the retry limit.
3. Parses and validates the JSON.
4. Removes exact duplicates.
5. Returns the valid records and a run summary.

The record count is capped at 100 per model in this starter notebook to keep Colab runs manageable. Increase it only after testing your runtime.


In [ ]:
def generate_dataset(
    model_label: str,
    description: str,
    strategy: str = "Broad scenario diversity",
    total_records: int = 10,
    temperature: float = 0.8,
    batch_size: int = 4,
    progress_callback=None,
):
    if model_label not in MODEL_CATALOG:
        raise ValueError(f"Unknown model selection: {model_label}")
    if not description or not description.strip():
        raise ValueError("Please enter a dataset description.")
    if not 1 <= int(total_records) <= 100:
        raise ValueError("Choose between 1 and 100 records per model.")

    model_id = MODEL_CATALOG[model_label]
    tokenizer, model = load_model(model_id)

    collected = []
    fingerprints = set()
    errors = []
    attempts = 0
    max_attempts = max(4, (int(total_records) // max(1, batch_size)) * 3 + 3)

    while len(collected) < int(total_records) and attempts < max_attempts:
        attempts += 1
        remaining = int(total_records) - len(collected)
        requested = min(int(batch_size), remaining)

        retry_note = ""
        if attempts > 1:
            retry_note = (
                f"Previous attempts did not produce enough valid unique records. "
                f"Generate {requested} NEW records with different scenarios and wording."
            )

        prompt = build_prompt(description, strategy, requested, retry_note)

        try:
            raw = generate_text(
                tokenizer, model, prompt,
                temperature=temperature,
                max_new_tokens=min(1000, 250 + requested * 220),
            )
            parsed = extract_json_array(raw)
            valid, validation_errors = validate_records(parsed)
            errors.extend(validation_errors)

            for record in valid:
                fingerprint = record_fingerprint(record)
                if fingerprint in fingerprints:
                    continue
                fingerprints.add(fingerprint)
                record["record_id"] = f"{model_label.split()[0].lower()}_{len(collected)+1:04d}"
                collected.append(record)
                if len(collected) >= int(total_records):
                    break

        except Exception as exc:
            errors.append(f"Attempt {attempts}: {type(exc).__name__}: {exc}")

        if progress_callback:
            progress_callback(
                f"{model_label}: {len(collected)}/{total_records} valid unique records "
                f"(attempt {attempts}/{max_attempts})"
            )

    summary = {
        "model": model_label,
        "model_id": model_id,
        "requested": int(total_records),
        "generated": len(collected),
        "attempts": attempts,
        "validation_errors": errors[:20],
    }

    return collected, summary


def records_to_dataframe(records, model_label=None):
    rows = []
    for record in records:
        row = dict(record)
        row["attributes"] = json.dumps(row["attributes"], ensure_ascii=False)
        if model_label:
            row["model"] = model_label
        rows.append(row)

    columns = ["record_id", "scenario", "text", "attributes", "label"]
    if model_label:
        columns.append("model")
    return pd.DataFrame(rows, columns=columns)


## 6. Evaluation helpers

These are basic engineering checks, not proof of real-world representativeness.

- **Schema validity:** whether records passed the required-key checks.
- **Exact duplicate rate:** repeated records after normalisation.
- **Label distribution:** counts by generated label.
- **Text length:** a simple diagnostic for unusually short/long examples.

Semantic diversity requires a stronger method (for example, embedding similarity) and should be added as a separate evaluation rather than inferred from exact duplicate removal.


In [ ]:
def evaluate_dataframe(df: pd.DataFrame) -> dict:
    if df.empty:
        return {
            "records": 0,
            "exact_duplicate_rate": 0.0,
            "average_text_characters": 0.0,
            "label_distribution": {},
        }

    text_norm = df["text"].fillna("").str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
    duplicate_count = int(text_norm.duplicated().sum())

    return {
        "records": int(len(df)),
        "exact_duplicate_rate": round(duplicate_count / max(1, len(df)), 4),
        "average_text_characters": round(float(df["text"].fillna("").str.len().mean()), 1),
        "label_distribution": df["label"].value_counts(dropna=False).to_dict(),
    }


def save_exports(df: pd.DataFrame, prefix="synthetic_dataset"):
    output_dir = Path("/content/synthetic_data_outputs")
    output_dir.mkdir(parents=True, exist_ok=True)

    csv_path = output_dir / f"{prefix}.csv"
    json_path = output_dir / f"{prefix}.json"

    df.to_csv(csv_path, index=False, encoding="utf-8-sig")

    json_records = df.to_dict(orient="records")
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(json_records, f, ensure_ascii=False, indent=2)

    return str(csv_path), str(json_path)


## 7. Build the Gradio application

Choose one model or compare all three. In comparison mode, models are loaded and run one after another to reduce memory use.

**Note:** The first request for a model can take longer because its weights must be downloaded and loaded.


In [ ]:
MODEL_CHOICES = list(MODEL_CATALOG.keys())
MODEL_CHOICES_WITH_COMPARE = ["Compare all models"] + MODEL_CHOICES

def run_gradio_generation(
    description,
    model_choice,
    strategy,
    record_count,
    temperature,
    batch_size,
):
    if not description or not description.strip():
        yield "Please enter a dataset description.", pd.DataFrame(), None, None, {}
        return

    selected_models = MODEL_CHOICES if model_choice == "Compare all models" else [model_choice]
    all_records = []
    summaries = []

    for model_label in selected_models:
        yield (
            f"Starting {model_label}. The model may need to download/load first...",
            pd.DataFrame(all_records),
            None,
            None,
            {"status": "running", "models_completed": len(summaries)},
        )

        try:
            records, summary = generate_dataset(
                model_label=model_label,
                description=description,
                strategy=strategy,
                total_records=int(record_count),
                temperature=float(temperature),
                batch_size=int(batch_size),
            )
            summaries.append(summary)

            for record in records:
                record["model"] = model_label
            all_records.extend(records)

        except Exception as exc:
            summaries.append({
                "model": model_label,
                "error": f"{type(exc).__name__}: {exc}",
            })

        # Release GPU memory between models, including in comparison mode.
        release_model()

        current_df = records_to_dataframe(all_records)
        yield (
            f"Finished {model_label}. Total valid records so far: {len(all_records)}",
            current_df,
            None,
            None,
            {"model_summaries": summaries},
        )

    final_df = records_to_dataframe(all_records)
    if final_df.empty:
        yield (
            "No valid records were generated. Review the model errors and try a smaller batch or another model.",
            final_df,
            None,
            None,
            {"model_summaries": summaries},
        )
        return

    csv_path, json_path = save_exports(final_df)
    evaluation = evaluate_dataframe(final_df)

    final_report = {
        "model_summaries": summaries,
        "overall_evaluation": evaluation,
        "important_note": "Synthetic records are fictional and require human review before use.",
    }

    yield (
        f"Complete. Generated {len(final_df)} valid records.",
        final_df,
        csv_path,
        json_path,
        final_report,
    )


with gr.Blocks(title="Multi-Model Synthetic Data Generator") as demo:
    gr.Markdown(
        "# Multi-Model Synthetic Data Generator\n"
        "Generate fictional structured datasets with Hugging Face instruction models."
    )

    with gr.Row():
        with gr.Column(scale=2):
            description = gr.Textbox(
                label="Describe the dataset you want",
                value=SOLAR_DEMO_PROMPT,
                lines=7,
                placeholder="Describe the business, records, attributes and intended use..."
            )
            model_choice = gr.Dropdown(
                choices=MODEL_CHOICES_WITH_COMPARE,
                value=MODEL_CHOICES[0],
                label="Model"
            )
            strategy = gr.Dropdown(
                choices=list(PROMPT_STRATEGIES.keys()),
                value="Broad scenario diversity",
                label="Prompt strategy"
            )
        with gr.Column(scale=1):
            record_count = gr.Slider(
                minimum=1, maximum=20, value=6, step=1,
                label="Records per model"
            )
            temperature = gr.Slider(
                minimum=0.1, maximum=1.2, value=0.8, step=0.1,
                label="Temperature"
            )
            batch_size = gr.Slider(
                minimum=1, maximum=4, value=2, step=1,
                label="Records requested per generation call"
            )

    generate_button = gr.Button("Generate synthetic data", variant="primary")
    status = gr.Textbox(label="Generation status", interactive=False)
    output_table = gr.Dataframe(
        label="Generated records",
        interactive=False,
        wrap=True
    )

    with gr.Row():
        csv_file = gr.File(label="Download CSV")
        json_file = gr.File(label="Download JSON")

    report = gr.JSON(label="Generation and evaluation report")

    generate_button.click(
        fn=run_gradio_generation,
        inputs=[
            description, model_choice, strategy,
            record_count, temperature, batch_size
        ],
        outputs=[status, output_table, csv_file, json_file, report],
    )

print("Launching Gradio...")
demo.queue().launch(share=True, debug=True)


## 8. How to use the application

1. Run the setup and import cells.
2. Run the model configuration and helper-function cells.
3. Run the Gradio cell.
4. Open the Gradio public link generated by Colab.
5. Keep the default solar prompt for your first test.
6. Start with one model, 6 records, batch size 2, and temperature 0.8.
7. Inspect the output and download CSV/JSON.
8. Then select **Compare all models**, using 3–5 records per model for the first comparison.\n9. Download your outputs before resetting the runtime.

The public Gradio share link is temporary. Do not put sensitive business information into a publicly shared demo.

## 9. Business applications and responsible use

Possible applications:
- **Solar:** fictional enquiries for testing lead routing, lead classification and customer-support workflows.
- **Aviation:** fictional maintenance narratives for testing document extraction and report classification. Do not use generated data as evidence of aircraft safety or failure rates.
- **Retail:** synthetic product reviews and service complaints for testing sentiment and routing.
- **Fintech:** fictional support conversations and transaction scenarios for software testing. Do not use generated examples alone to validate fraud detection.
- **Healthcare administration:** fictional appointment and administrative records. Do not treat generated records as clinically valid.

Before using synthetic data to train a model, assess:
- Whether generated labels are internally consistent.
- Whether the generated distribution resembles the intended real-world task.
- Whether the model has learned prompt artefacts rather than meaningful patterns.
- Whether synthetic data reproduces biases or memorised training examples.
- Performance on a separate, representative, appropriately governed real-world test set.

## 10. Troubleshooting and runtime resets

**If you need to reset the Colab runtime**
1. Download any CSV/JSON files you want to keep first; files under `/content` are temporary.
2. Choose **Runtime → Disconnect and delete runtime** (or **Restart session** for a clean process).
3. Reconnect and select the T4 GPU runtime again if available.
4. Run the notebook from the first cell. Model weights may need to download again.

A normal restart clears loaded models and GPU allocations. You do not need to change the notebook code after a reset.

## Troubleshooting

**CUDA out of memory**
- Restart the Colab runtime.
- Select one model instead of comparing all.
- Reduce `batch_size` to 1 or 2.
- Reduce `max_new_tokens` in `generate_dataset`.
- Use a smaller model.

**Model download or access error**
- Check the repository ID and internet connection.
- Check whether the model repository requires accepting terms or Hugging Face authentication.
- Try another model in `MODEL_CATALOG`.

**Invalid JSON or too few records**
- Reduce records per generation call.
- Use a lower record count for the first test.
- Try Qwen first, then compare other models.
- Inspect the generation errors in the final report.
- Increase the retry limit only after checking runtime time and token usage.

**Gradio link does not open**
- Confirm the final cell is still running.
- Use the `gradio.live` link printed by Colab.
- Re-run the Gradio cell after restarting the runtime.

---

### Submission checklist
- [ ] Demonstrate at least two models.
- [ ] Demonstrate at least two prompt strategies.
- [ ] Show the Gradio interface.
- [ ] Export a CSV and JSON dataset.
- [ ] Report the number of valid records and validation failures.
- [ ] Compare model outputs with examples.
- [ ] Explain one realistic business use case.
- [ ] State clearly that the generated records are synthetic, not real observations.
